In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Устанавливаем seed для воспроизводимости результатов
np.random.seed(42)

# ==========================================
# 1. Справочник номенклатуры и себестоимости (1C)
# ==========================================
products_data = {
    'SKU': ['UL-001', 'UL-002', 'UL-003', 'UL-004', 'UL-005', 'UL-006', 'UL-007'],
    'Item_Name': [
        'Сортер деревянный Радуга', 
        'Набор кубиков Классика (24 шт)', 
        'Развивающий Бизиборд Тучка',
        'Пазл-вкладыш Геометрия',
        'Деревянная пирамидка Овощи',
        'Магнитная рыбалка Котики',
        'Конструктор Монтессори Городок'
    ],
    'Category': ['Сортеры', 'Кубики', 'Бизиборды', 'Пазлы', 'Пирамидки', 'Игры', 'Конструкторы'],
    'Production_Cost': [320, 210, 850, 180, 150, 290, 640] # Себестоимость производства в рублях
}
df_production = pd.DataFrame(products_data)
df_production.to_csv('1C_Production_Cost.csv', index=False, encoding='utf-8-sig')

# ==========================================
# 2. Генерация Массива Продаж (200 000 строк)
# ==========================================
n_rows = 200000

# Генерируем временной ряд за последние полгода
start_date = datetime(2026, 1, 1)
dates = [start_date + timedelta(minutes=int(i)) for i in np.random.randint(0, 250000, n_rows)]

# Базовые параметры ритейла
marketplaces = ['Wildberries', 'Ozon', 'Yandex Market']
mp_weights = [0.55, 0.35, 0.10] # Имитируем реальные доли рынка: WB лидирует

regions = ['Центральный РФ', 'Приволжский РФ', 'Урал', 'Сибирь', 'Дальний Восток']
region_weights = [0.4, 0.3, 0.15, 0.1, 0.05]

# Генерируем базовые столбцы
df_sales = pd.DataFrame({
    'Date': sorted(dates),
    'Marketplace': np.random.choice(marketplaces, size=n_rows, p=mp_weights),
    'Region': np.random.choice(regions, size=n_rows, p=region_weights)
})

# Привязываем артикулы с разной вероятностью (какие-то игрушки покупают чаще)
sku_pool = products_data['SKU']
sku_weights = [0.25, 0.20, 0.05, 0.15, 0.15, 0.10, 0.10] # Бизиборд (UL-003) дорогой, его берут реже
df_sales['SKU'] = np.random.choice(sku_pool, size=n_rows, p=sku_weights)

# Добавляем базовые розничные цены через словарь соответствия
base_prices = {'UL-001': 1190, 'UL-002': 790, 'UL-003': 2990, 'UL-004': 590, 'UL-005': 490, 'UL-006': 890, 'UL-007': 1990}
df_sales['Retail_Price'] = df_sales['SKU'].map(base_prices)

# Вводим случайные колебания цены (акции, промо-скидки)
price_fluctuation = np.random.uniform(0.85, 1.0, size=n_rows) # скидки до 15%
df_sales['Retail_Price'] = (df_sales['Retail_Price'] * price_fluctuation).round(-1)

# Считаем количество штук в чеке (чаще 1 шт, реже 2 или 3)
df_sales['Qty'] = np.random.choice([1, 2, 3], size=n_rows, p=[0.85, 0.12, 0.03])

# Рассчитываем экономику маркетплейсов на основе бизнес-логики
# Комиссия зависит от площадки (условно: WB 15%, Ozon 12%, Yandex 10%)
mp_commission_pct = {'Wildberries': 0.15, 'Ozon': 0.12, 'Yandex Market': 0.10}
df_sales['Commission_Fee'] = df_sales.apply(lambda r: r['Retail_Price'] * r['Qty'] * mp_commission_pct[r['Marketplace']], axis=1).round(2)

# Логистика зависит от удаленности региона
region_logistics = {'Центральный РФ': 50, 'Приволжский РФ': 65, 'Урал': 90, 'Сибирь': 140, 'Дальний Восток': 220}
df_sales['Logistics_Cost'] = df_sales['Region'].map(region_logistics) * df_sales['Qty']

# Добавляем стоимость хранения (случайная величина, распределенная по позициям)
df_sales['Storage_Cost'] = np.random.uniform(2, 15, size=n_rows).round(2)

# 🔥 УМЫШЛЕННО ПОРТИМ ДАННЫЕ ДЛЯ ПОРТФОЛИО (Имитируем человеческий фактор)
# 1. Добавляем пробелы в артикулы в 5% строк
df_sales.loc[df_sales.sample(frac=0.05).index, 'SKU'] = df_sales['SKU'] + ' '
# 2. Переводим в нижний регистр в 5% строк
df_sales.loc[df_sales.sample(frac=0.05).index, 'SKU'] = df_sales['SKU'].str.lower()

# Сохраняем мега-файл продаж
df_sales.to_csv('Marketplaces_Raw_Sales.csv', index=False, encoding='utf-8-sig')

# ==========================================
# 3. Справочник Экспортных Поставок (ВЭД)
# ==========================================
n_export = 800
countries = ['Германия', 'Казахстан', 'Беларусь', 'Сербия', 'ОАЭ', 'Италия', 'Армения']
country_lead_time = {'Германия': 35, 'Казахстан': 12, 'Беларусь': 7, 'Сербия': 25, 'ОАЭ': 45, 'Италия': 40, 'Армения': 14}

export_dates = [start_date + timedelta(days=int(i)) for i in np.random.randint(0, 180, n_export)]

df_export = pd.DataFrame({
    'Invoice_Num': [f'EXP-{2026:02d}-{i:04d}' for i in range(n_export)],
    'Date': sorted(export_dates),
    'Country': np.random.choice(countries, size=n_export),
    'SKU': np.random.choice(sku_pool, size=n_export, p=sku_weights),
    'Batch_Qty': np.random.choice([100, 200, 500, 1000], size=n_export, p=[0.4, 0.3, 0.2, 0.1])
})

# Экспортные цены FOB в долларах
export_prices_usd = {'UL-001': 12, 'UL-002': 8, 'UL-003': 32, 'UL-004': 6, 'UL-005': 5, 'UL-006': 9, 'UL-007': 20}
df_export['Price_USD'] = df_export['SKU'].map(export_prices_usd)
df_export['Exchange_Rate_USD_RUB'] = np.random.uniform(91.0, 93.5, size=n_export).round(2)
df_export['Customs_and_Logistics_RUB'] = (df_export['Batch_Qty'] * np.random.uniform(30, 70, size=n_export)).round(0)
df_export['Lead_Time_Days'] = df_export['Country'].map(country_lead_time)

df_export.to_csv('Export_Shipments.csv', index
                =False, encoding='utf-8-sig')

print("🎉 Все 3 синтетических датасета успешно созданы и сохранены в CSV!")

🎉 Все 3 синтетических датасета успешно созданы и сохранены в CSV!
